# Notebook 02 — Pipeline de Prétraitement

Ce notebook construit le pipeline sklearn qui transforme les données brutes en features exploitables par les modèles. L'enjeu principal est d'éviter la **fuite de données** (data leakage) : si le `StandardScaler` apprend les statistiques sur l'ensemble complet avant le split, il "voit" indirectement les données de test, ce qui biaise les évaluations à la hausse.

**Ordre appliqué ici :**
1. `train_test_split` stratifié **en premier** — le test set est mis de côté définitivement
2. `Pipeline.fit()` **uniquement sur `X_train`** — le scaler, l'imputeur et l'encodeur apprennent sur le train
3. `Pipeline.transform()` appliqué séparément sur train et test — aucune information du test ne contaminate le train

In [1]:
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

from preprocessing import (
    load_raw_data, engineer_features,
    build_preprocessing_pipeline, get_feature_names_out,
    NUMERIC_FEATURES, CATEGORICAL_FEATURES, TARGET_CHURN,
)

df = load_raw_data('../data/customer_churn_business_dataset.csv')
df = engineer_features(df)
print(f'Shape après feature engineering: {df.shape}')
df.head(3)

Shape après feature engineering: (10000, 34)


,gender,age,country,city,customer_segment,tenure_months,signup_channel,contract_type,monthly_logins,weekly_active_days,...,escalations,email_open_rate,marketing_click_rate,nps_score,survey_response,referral_count,churn,tickets_per_tenure,fee_per_tenure,engagement_score
0,Male,68,Bangladesh,London,SME,22,Web,Monthly,26,7,...,0,0.71,0.40,27,Satisfied,1,0,0.173913,1.304348,0.549839
1,Female,57,Canada,Sydney,Individual,9,Mobile,Monthly,7,5,...,0,0.78,0.33,-19,Neutral,2,1,0.100000,3.000000,0.425641
2,Male,24,Germany,New York,SME,58,Web,Yearly,19,5,...,0,0.35,0.49,80,Neutral,1,0,0.016949,0.338983,0.510358


In [2]:
NUMERIC_ALL = NUMERIC_FEATURES + ['tickets_per_tenure', 'fee_per_tenure', 'engagement_score']
FEATURES    = NUMERIC_ALL + CATEGORICAL_FEATURES

X = df[FEATURES]
y = df[TARGET_CHURN].astype(int)

print(f'Distribution cible : {y.value_counts().to_dict()}')
print(f'Taux de churn global : {y.mean():.3f}')

Distribution cible : {0: 8979, 1: 1021}
Taux de churn global : 0.102


## Étape 1 — Split stratifié

Le split est stratifié sur la variable cible : les 10.2% de churners doivent se retrouver aussi bien dans le train que dans le test. Sans stratification, un split aléatoire pourrait créer un déséquilibre supplémentaire.

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,          # ratio churn préservé dans train ET test
    random_state=42,
)

print(f'Train : {len(X_train)} lignes | churn rate = {y_train.mean():.3f}')
print(f'Test  : {len(X_test)}  lignes | churn rate = {y_test.mean():.3f}')
print('→ Les taux de churn sont équivalents : le split est bien stratifié.')

Train : 8000 lignes | churn rate = 0.102
Test  : 2000  lignes | churn rate = 0.102
→ Les taux de churn sont équivalents : le split est bien stratifié.


## Étape 2 — Fit du pipeline sur X_train uniquement

Deux choix d'imputation à noter :
- **Variables numériques** → médiane (robuste aux valeurs extrêmes de `monthly_fee`)
- **`complaint_type`** → valeur constante `"Unknown"` plutôt que `most_frequent` : les 20% de NaN ne sont pas aléatoires, ils représentent les clients qui n'ont pas soumis de réclamation — une information en soi

In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])
categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('encoder', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False)),
])
preprocessor = ColumnTransformer([
    ('num', numeric_transformer, NUMERIC_ALL),
    ('cat', categorical_transformer, CATEGORICAL_FEATURES),
], remainder='drop')

# FIT SUR X_TRAIN UNIQUEMENT
X_train_proc = preprocessor.fit_transform(X_train)
# TRANSFORM SUR X_TEST (le scaler utilise les stats de X_train, pas de X_test)
X_test_proc  = preprocessor.transform(X_test)

print(f'X_train_proc shape : {X_train_proc.shape}')
print(f'X_test_proc  shape : {X_test_proc.shape}')

X_train_proc shape : (8000, 50)
X_test_proc  shape : (2000, 50)


In [5]:
# Vérification : les moyennes du scaler sont celles du TRAIN, pas du dataset complet
scaler = preprocessor.named_transformers_['num'].named_steps['scaler']
print('Moyenne du scaler (appris sur X_train) :')
for name, mean in zip(NUMERIC_ALL, scaler.mean_):
    print(f'  {name:30s} : {mean:.4f}')

print('\n→ Ces valeurs correspondent aux stats du train set, pas du dataset complet.')
print('  Si elles correspondaient au dataset complet, il y aurait fuite de données.')

Moyenne du scaler (appris sur X_train) :
  age                            : 45.9472
  tenure_months                  : 30.2529
  monthly_logins                 : 19.7318
  weekly_active_days             : 3.4937
  avg_session_time               : 15.2518
  features_used                  : 5.0075
  usage_growth_rate              : 0.0198
  last_login_days_ago            : 9.4911
  monthly_fee                    : 34.8800
  total_revenue                  : 1055.9075
  payment_failures               : 0.5014
  support_tickets                : 1.2089
  avg_resolution_time            : 23.9541
  csat_score                     : 3.4864
  escalations                    : 0.2921
  email_open_rate                : 0.5010
  marketing_click_rate           : 0.2544
  nps_score                      : 19.0550
  referral_count                 : 0.9909
  tickets_per_tenure             : 0.0756
  fee_per_tenure                 : 2.1734
  engagement_score               : 0.4310

→ Ces valeurs correspond

## Récupération des noms de features

Après OneHotEncoding, les noms de colonnes changent (`contract_type` → `contract_type_Yearly`, `contract_type_Quarterly`…). On les récupère ici pour que les graphiques d'importance et les valeurs SHAP restent lisibles.

In [6]:
ohe = preprocessor.named_transformers_['cat'].named_steps['encoder']
feature_names = NUMERIC_ALL + list(ohe.get_feature_names_out(CATEGORICAL_FEATURES))
print(f'Features après transformation ({len(feature_names)}) :')
print(feature_names)

Features après transformation (50) :
['age', 'tenure_months', 'monthly_logins', 'weekly_active_days', 'avg_session_time', 'features_used', 'usage_growth_rate', 'last_login_days_ago', 'monthly_fee', 'total_revenue', 'payment_failures', 'support_tickets', 'avg_resolution_time', 'csat_score', 'escalations', 'email_open_rate', 'marketing_click_rate', 'nps_score', 'referral_count', 'tickets_per_tenure', 'fee_per_tenure', 'engagement_score', 'gender_Male', 'country_Bangladesh', 'country_Canada', 'country_Germany', 'country_India', 'country_UK', 'country_USA', 'city_Delhi', 'city_Dhaka', 'city_London', 'city_New York', 'city_Sydney', 'city_Toronto', 'customer_segment_Individual', 'customer_segment_SME', 'signup_channel_Referral', 'signup_channel_Web', 'contract_type_Quarterly', 'contract_type_Yearly', 'payment_method_Card', 'payment_method_PayPal', 'discount_applied_Yes', 'price_increase_last_3m_Yes', 'complaint_type_Service', 'complaint_type_Technical', 'complaint_type_Unknown', 'survey_resp

In [7]:
# Sauvegarder pour les notebooks suivants
import joblib
from pathlib import Path

Path('../models').mkdir(exist_ok=True)
joblib.dump(preprocessor, '../models/preprocessor_churn.joblib')
joblib.dump({
    'X_train': X_train_proc, 'X_test': X_test_proc,
    'y_train': y_train.values, 'y_test': y_test.values,
    'feature_names': feature_names,
    'X_train_raw': X_train, 'X_test_raw': X_test,
}, '../models/processed_data.joblib')

print('Pipeline et données prétraitées sauvegardés dans models/')
print('→ Prochaine étape : 03_modeling_churn.ipynb')

Pipeline et données prétraitées sauvegardés dans models/
→ Prochaine étape : 03_modeling_churn.ipynb
